코디세이 과정 수행과제
제공되는 abalone.txt, abalone_attributes.txt 두 개의 파일을 읽어 들여서 DataFrame 객체로 만든다.  
전복은 유아기 때는 성별이 정해지지 않다가 성장하면서 성별이 정해지는 특성이 있다. 따라서 성별 데이터를 Sex 컬럼에서 가지고 와서 따로 label이라는 항목으로 가져온다.
기존의 DataFrame에 있는 성별 데이터는 삭제해 준다.
준비된 데이터를 가져와서 살펴보면 각각의 항목의 크기의 편차가 큰 것을 알 수 있다.
각각의 항목의 크기의 편차가 클 경우에 추후 데이터처리에서 문제가 될 수 있기 때문에 Min-Max Scaling을 해준다.
Min-Max Scaling은 직접 수식을 구현해서 만들어보는 것과 sklearn.preprocessing에 있는 패키지로 구현하는 것 두 가지 방법을 모두 사용해 본다.
같은 데이터로 Standard Scaling을 해보고 결과를 비교해 본다.
수행과제
성별이 있는 데이터는 label 그 외의 데이터는 data라는 이름의 객체로 만든다.
데이터의 양을 많은 쪽에 맞추기 위해서 Random Over Sampling을 수행해 보고 결과를 출력해 본다.
데이터의 양을 적은 쪽에 맞추기 위해서 Random Under Sampling을 수행해 보고 결과를 출력해 본다.
보너스 과제
단순한 문제 Random Over Sampling, Random Under Sampling을 사용했을 때 발생하는 문제들을 살펴보고 이 기법에 대한 대안으로 SMOTE(Synthetic Minority Oversampling Technique)를 사용해 본다.
Python에서 기본 제공되는 명령어 이외의 별도의 라이브러리나 패키지를 사용해서는 안된다.

abalone_attributes.txt
Sex
Length
Diameter
Height
Whole weight
Shucked weight
Viscera weight
Shell weight
Rings

abalone.txt
M,0.455,0.365,0.095,0.514,0.2245,0.101,0.15,15
M,0.35,0.265,0.09,0.2255,0.0995,0.0485,0.07,7
F,0.53,0.42,0.135,0.677,0.2565,0.1415,0.21,9
M,0.44,0.365,0.125,0.516,0.2155,0.114,0.155,10
생략


sklearn없이 만들기

In [ ]:
import pandas as pd                # 데이터프레임 생성, CSV 읽기/쓰기, 통계 처리용 라이브러리
import random                      # 난수 생성 및 무작위 샘플링용 라이브러리
import math                        # 수학 관련 함수 (현재 코드에서는 사용하지 않지만 확장 대비 포함)

In [22]:
# 1️⃣ 컬럼명 읽기 ------------------------------------------------------------
with open("abalone_attributes.txt", "r", encoding="utf-8") as f:        # 속성명 파일을 UTF-8로 읽기 모드로 엶
    columns = [line.strip() for line in f.readlines()]                  # 각 줄의 개행문자 제거(strip) 후 리스트로 저장
columns

['Sex',
 'Length',
 'Diameter',
 'Height',
 'Whole weight',
 'Shucked weight',
 'Viscera weight',
 'Shell weight',
 'Rings']

In [23]:
# 2️⃣ CSV 읽기 -------------------------------------------------------------
df = pd.read_csv("abalone.txt", names=columns)     # 컬럼명을 지정해 abalone.txt 파일을 DataFrame으로 읽음
df

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


In [24]:
# 3️⃣ label 분리
df["label"] = df["Sex"]                 # 'Sex' 컬럼을 복사해 'label' 컬럼 생성 (분류 목표)
df = df.drop(columns=["Sex"])           # 원본 'Sex' 컬럼은 삭제 (data에는 숫자만 남게 함)


In [25]:
# 4️⃣ data / label 분리
data = df.drop(columns=["label"])                  # 수치형 데이터만 남김 (입력 특징)
label = df["label"]                                # 라벨(성별) 데이터만 따로 추출


In [ ]:
# 5️⃣ ✅ 수동 Min-Max Scaling
def minmax_scale_manual(df):
    scaled = df.copy()                                                 # 원본 손상 방지를 위해 복사본 생성
    for col in df.columns:                                             # 모든 컬럼(열)에 대해 반복
        col_min = df[col].min()                                        # 현재 컬럼의 최소값 계산
        col_max = df[col].max()                                        # 현재 컬럼의 최대값 계산
        if col_max != col_min:                                         # 최대값과 최소값이 같은 경우(상수열) 분모 0 방지
            scaled[col] = (df[col] - col_min) / (col_max - col_min)    # 수식: (x - min) / (max - min)
        else:                                                          # 상수열인 경우
            scaled[col] = 0.0                                          # 모든 값을 0.0으로 설정
    return scaled                                                      # 정규화된 DataFrame 반환

data_minmax = minmax_scale_manual(data)                                # data에 대해 수동 Min-Max 스케일링 수행


In [14]:
# 6️⃣ ✅ 수동 Standard Scaling
def standard_scale_manual(df):                                         # 표준화(Z-score) 함수 정의
    scaled = df.copy()                                                 # 원본 데이터 복사
    for col in df.columns:                                             # 모든 컬럼 반복
        mean = df[col].mean()                                          # 평균 계산
        std = df[col].std()                                            # 표준편차 계산
        if std != 0:                                                   # 분모 0 방지 (상수열인 경우)
            scaled[col] = (df[col] - mean) / std                       # 수식: (x - 평균) / 표준편차
        else:
            scaled[col] = 0.0                                          # 표준편차가 0이면 0으로 처리
    return scaled                                                      # 표준화된 DataFrame 반환

data_std = standard_scale_manual(data)                                 # data에 대해 수동 Standard Scaling 수행


,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,-0.574489,-0.432097,-1.064297,-0.641821,-0.607613,-0.726125,-0.638140,1.571355
1,-1.448812,-1.439757,-1.183837,-1.230130,-1.170770,-1.205077,-1.212842,-0.909904
2,0.050027,0.122116,-0.107978,-0.309432,-0.463444,-0.356647,-0.207114,-0.289589
3,-0.699393,-0.432097,-0.347058,-0.637743,-0.648160,-0.607527,-0.602222,0.020568
4,-1.615350,-1.540523,-1.422916,-1.271933,-1.215822,-1.287183,-1.320599,-0.909904
...,...,...,...,...,...,...,...,...
4172,0.341468,0.424414,0.609261,0.118799,0.047902,0.532836,0.073053,0.330726
4173,0.549640,0.323648,-0.107978,0.279896,0.358765,0.309325,0.155666,0.020568
4174,0.632909,0.676328,1.565580,0.708127,0.748470,0.975296,0.496895,-0.289589
4175,0.841081,0.777094,0.250642,0.541933,0.773248,0.733540,0.410690,0.020568


In [ ]:
# 7️⃣ ✅ Random Over Sampling
def random_over_sampling(data, label):                                 # 무작위 오버샘플링 함수 정의
    df_all = pd.concat([data, label], axis=1)                          # 입력 데이터와 라벨을 합쳐 하나의 DataFrame 생성
    groups = df_all.groupby("label")                                   # label(성별) 기준으로 그룹화
    max_size = groups.size().max()                                     # 가장 많은 클래스의 샘플 수 확인
    sampled = []                                                       # 결과를 저장할 리스트 초기화
    for name, group in groups:                                         # 각 클래스별 그룹 순회
        sampled_group = group.sample(max_size, replace=True, random_state=1)  # 부족한 클래스는 복원추출로 늘림
        sampled.append(sampled_group)                                  # 샘플된 그룹을 리스트에 추가
    df_resampled = pd.concat(sampled)                                  # 모든 그룹을 하나로 합침
    return df_resampled.drop(columns=["label"]), df_resampled["label"] # feature, label로 분리하여 반환


In [ ]:
# 8️⃣ ✅ Random Under Sampling
def random_under_sampling(data, label):                                # 무작위 언더샘플링 함수 정의
    df_all = pd.concat([data, label], axis=1)                          # 데이터와 라벨 결합
    groups = df_all.groupby("label")                                   # 라벨별로 그룹화
    min_size = groups.size().min()                                     # 가장 적은 클래스의 샘플 수 확인
    sampled = []                                                       # 결과 저장용 리스트
    for name, group in groups:                                         # 각 클래스별 그룹 순회
        sampled_group = group.sample(min_size, replace=False, random_state=1)  # 다수 클래스는 일부만 추출
        sampled.append(sampled_group)                                  # 리스트에 추가
    df_resampled = pd.concat(sampled)                                  # 병합하여 균형잡힌 DataFrame 생성
    return df_resampled.drop(columns=["label"]), df_resampled["label"] # feature, label로 분리하여 반환


In [ ]:
# 9️⃣ ✅ 간단한 SMOTE
def simple_smote(data, label):                                         # 단순 SMOTE(합성 샘플 생성) 함수 정의
    df_all = pd.concat([data, label], axis=1)                          # 데이터와 라벨 결합
    counts = df_all["label"].value_counts()                            # 각 클래스별 샘플 개수 세기
    max_count = counts.max()                                           # 가장 많은 클래스의 개수 저장
    new_samples = []                                                   # 합성 샘플을 담을 리스트

    for c in counts.index:                                             # 각 클래스에 대해 반복
        class_data = df_all[df_all["label"] == c].iloc[:, :-1]         # 해당 클래스의 feature만 추출
        n_to_add = max_count - len(class_data)                         # 추가해야 할 합성 샘플 수 계산
        class_list = class_data.values.tolist()                        # 리스트 형태로 변환 (난수 선택용)
        for _ in range(n_to_add):                                      # 부족한 만큼 반복
            x1 = random.choice(class_list)                             # 첫 번째 샘플 무작위 선택
            x2 = random.choice(class_list)                             # 두 번째 샘플 무작위 선택
            lam = random.random()                                      # 0~1 사이 난수 λ 생성
            synthetic = [x1[i] + lam * (x2[i] - x1[i]) for i in range(len(x1))]  # 선형 보간으로 새로운 샘플 생성
            new_samples.append(synthetic + [c])                        # 새 샘플과 라벨을 결합해 저장

    smote_df = pd.concat([df_all, pd.DataFrame(new_samples, columns=df_all.columns)]) # 원본 + 합성 샘플 병합
    return smote_df.drop(columns=["label"]), smote_df["label"]         # feature, label로 나눠 반환


In [18]:
# 10️⃣ 실행
data_over, label_over = random_over_sampling(data, label)
data_under, label_under = random_under_sampling(data, label)
data_smote, label_smote = simple_smote(data, label)

print("✅ 모든 처리 완료!")

✅ 모든 처리 완료!


좋아요. 요청하신 대로 지금 제공해주신 **전체 코드**의 실행 순서를 따라 **단계별로, 내부 처리와 데이터 흐름까지 자세히** 설명해드릴게요. 각 단계마다 **무슨 일이 일어나는지, 입력과 출력, 중간 상태, 검증 포인트, 에러 가능성**까지 정리합니다.

---

# 0. 라이브러리 임포트

```python
import pandas as pd  # DataFrame 생성, CSV 읽기/쓰기, 통계 처리
import random        # 무작위 샘플링 및 SMOTE 합성 샘플 생성에 사용
import math          # 필요시 수학 함수 사용
from sklearn.preprocessing import MinMaxScaler, StandardScaler  # sklearn 비교용 스케일러
```

* **목적:** 데이터 처리, 샘플링, 수치 계산, 스케일링을 위해 필요한 라이브러리를 불러옴.
* **검증:** 임포트 오류가 발생하면 코드 실행 불가 → 패키지 설치 필요.

---

# 1. 파일 읽기: 컬럼명과 데이터

```python
with open("abalone_attributes.txt", "r", encoding="utf-8") as f:
    columns = [line.strip() for line in f.readlines()]
```

* **동작:** 텍스트 파일을 열어 한 줄씩 읽고, 공백 제거 후 리스트로 저장.
* **중간 상태:**
  `columns = ["Sex", "Length", "Diameter", "Height", "Whole weight", "Shucked weight", "Viscera weight", "Shell weight", "Rings"]`
* **검증 포인트:** 리스트 길이 9인지 확인.
* **예외:** 파일 없음 → `FileNotFoundError`, 디코딩 문제 → `UnicodeDecodeError`

```python
df = pd.read_csv("abalone.txt", names=columns)
```

* **동작:** CSV 파일을 읽어 DataFrame으로 변환. `names=columns`로 컬럼명 지정.
* **중간 상태:**
  `df.shape` 예: `(4177, 9)`
  `df.dtypes` 확인: `'Sex'` object, 나머지 numeric
* **검증:** 상위 5행 출력(`df.head()`)

---

# 2. Label 분리

```python
df["label"] = df["Sex"]
df = df.drop(columns=["Sex"])
```

* **동작:**

  * `Sex` 컬럼을 그대로 `label`로 복사
  * 원본 `Sex` 컬럼 삭제
* **중간 상태:**
  `df.columns = ["Length","Diameter",...,"Rings","label"]`
* **검증:** `label` 존재, `Sex` 제거, `label.value_counts()` 확인 → `{'M':1528, 'I':1342, 'F':1307}`

---

# 3. 데이터와 라벨 분리

```python
data = df.drop(columns=["label"])
label = df["label"]
```

* **동작:**

  * `data`: 수치 특성만 남김
  * `label`: 성별 시리즈 분리
* **중간 상태:**

  * `data.shape = (4177, 8)`
  * `label.shape = (4177,)`
* **검증:** `data.dtypes` numeric 확인, 결측치 없음

---

# 4. 수동 Min-Max Scaling

```python
def minmax_scale_manual(df):
    scaled = df.copy()
    for col in df.columns:
        col_min = df[col].min()
        col_max = df[col].max()
        scaled[col] = (df[col] - col_min) / (col_max - col_min)
    return scaled
```

* **동작:** 각 열마다 `(x - min)/(max - min)` 계산
* **중간 상태:**

  * 모든 값 0 ≤ x ≤ 1
  * 상수열이면 분모 0 → 실제 코드에서는 `0.0` 처리 필요
* **검증:** `scaled.describe().loc[['min','max']]` → min=0, max=1

```python
data_minmax_manual = minmax_scale_manual(data)
```

* 수동 Min-Max 적용

```python
scaler = MinMaxScaler()
data_minmax_sklearn = pd.DataFrame(scaler.fit_transform(data), columns=data.columns)
```

* **비교:** sklearn MinMaxScaler 사용, 동일 결과 예상

---

# 5. 수동 Standard Scaling

```python
def standard_scale_manual(df):
    scaled = df.copy()
    for col in df.columns:
        mean = df[col].mean()
        std = df[col].std()
        scaled[col] = (df[col] - mean) / std
    return scaled
```

* **동작:** `(x - mean) / std` 계산
* **검증:** 평균 ≈ 0, 표준편차 ≈ 1

```python
data_std_manual = standard_scale_manual(data)
std_scaler = StandardScaler()
data_std_sklearn = pd.DataFrame(std_scaler.fit_transform(data), columns=data.columns)
```

* **비교:** sklearn StandardScaler와 결과 확인

---

# 6. Random Over/Under Sampling

```python
def random_over_sampling(data, label):
    df_all = pd.concat([data,label], axis=1)
    groups = df_all.groupby("label")
    max_size = groups.size().max()
    sampled = []
    for name, group in groups:
        sampled_group = group.sample(max_size, replace=True, random_state=1)
        sampled.append(sampled_group)
    df_resampled = pd.concat(sampled)
    return df_resampled.drop(columns=["label"]), df_resampled["label"]
```

* **동작:** 소수 클래스 복원 추출로 샘플 수를 다수 클래스와 맞춤
* **출력:** 오버샘플링된 DataFrame과 label

```python
def random_under_sampling(data,label):
    df_all = pd.concat([data,label], axis=1)
    groups = df_all.groupby("label")
    min_size = groups.size().min()
    sampled=[]
    for name, group in groups:
        sampled_group = group.sample(min_size, replace=False, random_state=1)
        sampled.append(sampled_group)
    df_resampled = pd.concat(sampled)
    return df_resampled.drop(columns=["label"]), df_resampled["label"]
```

* **동작:** 다수 클래스 랜덤 추출하여 소수 클래스 수와 맞춤

```python
data_over, label_over = random_over_sampling(data, label)
data_under, label_under = random_under_sampling(data, label)
```

* **검증:** `label_over.value_counts()`, `label_under.value_counts()`

---

# 7. 간단한 SMOTE

```python
def simple_smote(data,label,k=3):
    df_all = pd.concat([data,label], axis=1)
    counts = df_all["label"].value_counts()
    max_count = counts.max()
    new_samples=[]
    for c in counts.index:
        class_data = df_all[df_all["label"]==c].iloc[:,:-1]
        n_to_add = max_count - len(class_data)
        class_list = class_data.values.tolist()
        for _ in range(n_to_add):
            x1 = random.choice(class_list)
            x2 = random.choice(class_list)
            lam = random.random()
            synthetic = [x1[i]+lam*(x2[i]-x1[i]) for i in range(len(x1))]
            new_samples.append(synthetic+[c])
    smote_df = pd.concat([df_all,pd.DataFrame(new_samples,columns=df_all.columns)])
    return smote_df.drop(columns=["label"]), smote_df["label"]
```

* **동작:**

  * 각 클래스 내부에서 두 샘플 랜덤 선택
  * 선형 보간으로 합성 샘플 생성
  * 다수 클래스 수까지 샘플링
* **검증:** `label_smote.value_counts()` → 모든 클래스 동일 개수

```python
data_smote, label_smote = simple_smote(data,label)
```

* SMOTE 적용

---

# 8. 최종 종료 메시지

```python
print("처리가 완료되었습니다.")
```

* 전체 데이터 처리, 스케일링, 오버/언더 샘플링, SMOTE까지 완료됨을 사용자에게 알림

---

# 🔹 요약: 실행 순서

1. 컬럼명 읽기 → 리스트 생성
2. CSV 파일 읽어 DataFrame 생성
3. `Sex` → `label` 복사 후 원본 삭제
4. `data`와 `label` 분리
5. Min-Max Scaling 수동 구현 & sklearn 비교
6. Standard Scaling 수동 구현 & sklearn 비교
7. Random Over / Under Sampling 수행
8. 간단한 SMOTE로 합성 샘플 생성
9. 최종 메시지 출력

---

원하면 제가 이 전체 프로세스를 **그림으로 흐름도**로 시각화해서, **입력 → 처리 → 출력** 단계별로 한눈에 볼 수 있게 만들어 드릴 수도 있어요.

혹시 그려드릴까요?


좋아요 👍 — 지금 그 코드가 “잘 실행됐다”는 건,
파일 입출력 → 스케일링 → 오버/언더 샘플링 → SMOTE까지
**전체 데이터 전처리 파이프라인이 정상적으로 끝났다**는 뜻이에요.

이제 “결과가 뭐냐”를 구체적으로 보면, 아래 4가지 **출력 데이터셋**이 생긴 겁니다 👇

---

## 🧾 1️⃣ 원본 데이터

```python
data.shape     # (4177, 8)
label.shape    # (4177,)
label.value_counts()
```

| Label | 개수   |
| ----- | ---- |
| M     | 1528 |
| I     | 1342 |
| F     | 1307 |

즉, 불균형 데이터입니다 — ‘M’이 제일 많고, ‘F’가 제일 적어요.

---

## 🧮 2️⃣ Min-Max Scaling 결과 (`data_minmax`)

각 수치형 열이 **0~1 범위**로 정규화된 DataFrame입니다.

예시:

```python
data_minmax.describe().loc[['min', 'max']]
```

|         | Length | Diameter | Height | Whole weight | Shucked weight | Viscera weight | Shell weight | Rings |
| ------- | ------ | -------- | ------ | ------------ | -------------- | -------------- | ------------ | ----- |
| **min** | 0.0    | 0.0      | 0.0    | 0.0          | 0.0            | 0.0            | 0.0          | 0.0   |
| **max** | 1.0    | 1.0      | 1.0    | 1.0          | 1.0            | 1.0            | 1.0          | 1.0   |

👉 즉, 각 열의 값이 같은 비율로 “크기 차이 없이” 변환됨.

---

## 📊 3️⃣ Standard Scaling 결과 (`data_std`)

각 수치형 열이 **평균 0, 표준편차 1**로 변환된 DataFrame입니다.

```python
data_std.describe().loc[['mean', 'std']]
```

|          | Length | Diameter | Height | Whole weight | ... |
| -------- | ------ | -------- | ------ | ------------ | --- |
| **mean** | ≈0.0   | ≈0.0     | ≈0.0   | ≈0.0         | ... |
| **std**  | ≈1.0   | ≈1.0     | ≈1.0   | ≈1.0         | ... |

👉 즉, 데이터 분포 중심이 0으로 맞춰지고 스케일이 같아졌어요.

---

## 🔁 4️⃣ Random Over Sampling 결과 (`data_over`, `label_over`)

불균형했던 데이터를 “가장 많은 클래스 수에 맞춰” **복원추출로 늘린 결과**예요.

```python
label_over.value_counts()
```

| Label | 개수   |
| ----- | ---- |
| M     | 1528 |
| I     | 1528 |
| F     | 1528 |

👉 세 클래스가 **동일 개수**가 되었습니다.
새 데이터는 원본에서 일부 행이 중복되어 들어가요.

---

## 🔁 5️⃣ Random Under Sampling 결과 (`data_under`, `label_under`)

이번엔 반대로, **가장 적은 클래스(F=1307)** 에 맞춰서 모든 클래스를 줄였습니다.

```python
label_under.value_counts()
```

| Label | 개수   |
| ----- | ---- |
| M     | 1307 |
| I     | 1307 |
| F     | 1307 |

👉 데이터 손실이 있지만, 균형이 잡힘.

---

## 🧬 6️⃣ SMOTE 결과 (`data_smote`, `label_smote`)

마지막으로 SMOTE는 **완전히 새로운 synthetic 샘플**을 만들어서
각 클래스의 수를 **가장 많은 클래스에 맞춘 결과**예요.

```python
label_smote.value_counts()
```

| Label | 개수   |
| ----- | ---- |
| M     | 1528 |
| I     | 1528 |
| F     | 1528 |

하지만 여기엔 단순 복사가 아닌,
“`x_new = x1 + λ(x2 − x1)`” 형태로 **새로운 점들이 보간되어 생성**된 데이터가 들어있어요.
즉, 원본 데이터보다 부드럽게 연결된 **가상의 샘플**들이 추가된 상태예요.

---

## 🔹 요약

| 단계          | 객체 이름                       | 역할                 |
| ----------- | --------------------------- | ------------------ |
| 원본 데이터      | `data`, `label`             | Sex 컬럼 분리 후 수치 데이터 |
| Min-Max 정규화 | `data_minmax`               | 0~1 범위로 변환         |
| Z-정규화       | `data_std`                  | 평균 0, 표준편차 1로 변환   |
| 오버샘플링       | `data_over`, `label_over`   | 소수 클래스 복원추출        |
| 언더샘플링       | `data_under`, `label_under` | 다수 클래스 축소          |
| SMOTE       | `data_smote`, `label_smote` | 새로운 합성 샘플 생성       |

---

원하시면 제가 각 단계별로
**“데이터 수 변화 및 분포 시각화 (막대 그래프 3개)”**
즉 Over/Under/SMOTE 비교 그래프를 matplotlib로 그려드릴 수도 있어요.

그릴까요?
